In [ ]:
import anndata
import numpy as np
from scipy import sparse
import os
import numpy as np
import scanpy as sc
import decoupler as dc
import os
import pandas as pd
import decoupler as dc
import matplotlib.pyplot as plt
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import gseapy as gp
from scipy.stats import mannwhitneyu

In [ ]:
metadata = pd.read_csv(os.path.expanduser("~/Desktop/reznik/bodycomp_main/revision/analysis/metadatapdacpolar.csv"), index_col=0)
path = os.path.expanduser('~/Desktop/reznik/bodycomp_main/data/rnaseq/pancreatic/normalized_data_with_hvg_pca_umap_phenotypes_finalV12.h5ad')
adata = sc.read_h5ad(path)

In [ ]:
adata_filt = adata = adata[adata.obs.index.isin(metadata.index)]
adata_filt.obs = adata_filt.obs.join(metadata)

In [ ]:
adata_filt = adata_filt[adata_filt.obs['predicted_doublet'] == False]
sc.pp.filter_cells(adata_filt, min_genes=200)
sc.pp.filter_cells(adata_filt, min_counts=500)
sc.pp.filter_genes(adata_filt, min_cells=3)

In [ ]:
downloads_path = os.path.expanduser("~/Desktop/reznik/bodycomp_main/revision/figures/polar_pdac/")
sc.settings.figdir = downloads_path
sc.settings.dpi = 300
plt.rcParams['figure.figsize'] = (2, 2)
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 6  
plt.rcParams['axes.titlesize'] = 7  
plt.rcParams['pdf.fonttype'] = 42 

sc.pl.umap(adata_filt, color='cluster',
           size=0.5,   
            palette={'Type A': '#B07AA1FF', 'Type B': '#499894FF','Type C': '#A0CBE8FF', 'No Cachexia': '#FF9D9AFF'},       
    alpha=0.8,            frameon=False, 
    title = "Subtype",
    save="pdac_subtype.pdf")

In [ ]:

sc.settings.figdir = downloads_path
plt.rcParams['figure.figsize'] = (2, 2)
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 6
plt.rcParams['axes.titlesize'] = 7
plt.rcParams['pdf.fonttype'] = 42

ax = sc.pl.umap(
    adata_filt, color='cluster',
    size=0.5,
    palette={'Type A': '#B07AA1FF', 'Type B': '#499894FF',
             'Type C': '#A0CBE8FF', 'No Cachexia': '#FF9D9AFF'},
    alpha=0.8,
    frameon=False,
    title="Subtype",
    legend_fontsize=6,  
    show=False,         
)

ax.set_title(ax.get_title(), fontweight='bold')  

for coll in ax.collections:
    coll.set_rasterized(True)

fig = ax.figure
fig.savefig(f"{downloads_path}/pdac_subtype.pdf", dpi=300, bbox_inches='tight')
fig.savefig(f"{downloads_path}/pdac_subtype.png", dpi=600, bbox_inches='tight')

In [ ]:
plt.rcParams['figure.figsize'] = (2, 2)

ax = sc.pl.umap(
    adata_filt, color='auto_cell_type_V9',
    size=0.5,
    alpha=0.5,
    frameon=False,
    title="Cell Type",
    legend_fontsize=6,
    show=False,
)

ax.set_title(ax.get_title(), fontweight='bold')

for coll in ax.collections:
    coll.set_rasterized(True)

leg = ax.get_legend()
for handle in leg.legend_handles:   
    try:
        handle.set_markersize(3)
    except AttributeError:
        handle.set_sizes([10])

fig = ax.figure
fig.savefig(f"{downloads_path}/pdac_celltype.pdf", dpi=300, bbox_inches='tight')
fig.savefig(f"{downloads_path}/pdac_celltype.png", dpi=600, bbox_inches='tight')
plt.close()

In [ ]:
print(adata_filt.obs_keys)

In [ ]:
sc.pl.violin(
    adata_filt,
    ["n_genes_by_counts", "total_counts", "mito_frac"],
    jitter=0.4,
    multi_panel=True)

In [ ]:
sc.pl.umap(
    adata_filt,
    color=["log1p_total_counts", "mito_frac", "log1p_n_genes_by_counts"],
    wspace=0.5,
    ncols=2,
)

In [ ]:
sc.pl.scatter(adata_filt, "total_counts", "n_genes_by_counts", color="mito_frac")

In [ ]:
pdata = dc.pp.pseudobulk(
    adata_filt,
    sample_col="patient_ID",
    groups_col="auto_cell_type_V9",
    layer="raw",     
    mode="sum"
)

In [ ]:

dc.pl.filter_samples(
    adata=pdata,
    groupby=["cluster", "patient_ID", "auto_cell_type_V9"],  
    min_cells = 10, 
    min_counts = 1000,
    figsize=(5, 8),
)

In [ ]:
dc.pp.filter_samples(pdata, min_cells=10, min_counts=1000)

In [ ]:
results = {}

outdir = os.path.expanduser("~/Desktop/reznik/bodycomp_main/revision/results/polar_scrna/degs/")
os.makedirs(outdir, exist_ok=True)
print(outdir)
# All pairwise comparisons you want to run
contrasts = [
    ("Type A", "Type C"),
]

min_samples = 2  # minimum samples per group to attempt a contrast

for celltype in pdata.obs["auto_cell_type_V9"].unique():

    print(f"\nRunning DESeq2 for cell type: {celltype}")

    adata_sub = pdata[pdata.obs["auto_cell_type_V9"] == celltype].copy()
    

    counts = adata_sub.obs["cluster"].value_counts()
    available_groups = set(adata_sub.obs["cluster"].unique())

    print(counts)

    # Which contrasts are actually feasible for this cell type?
    feasible_contrasts = [
        (g1, g2) for g1, g2 in contrasts
        if g1 in available_groups and g2 in available_groups
        and counts.get(g1, 0) >= min_samples
        and counts.get(g2, 0) >= min_samples
    ]

    if not feasible_contrasts:
        print(f"Skipping {celltype}: no feasible contrasts (insufficient group sizes)")
        continue

    # Only keep samples belonging to groups involved in at least one feasible contrast
    groups_needed = set(g for pair in feasible_contrasts for g in pair)
    adata_sub = adata_sub[adata_sub.obs["cluster"].isin(groups_needed)].copy()

    # drop unused categories left over from subsetting (avoids empty design matrix columns)
    adata_sub.obs["cluster"] = adata_sub.obs["cluster"].astype("category").cat.remove_unused_categories()
    adata_sub.obs["patient_ID"] = adata_sub.obs["patient_ID"].astype("category").cat.remove_unused_categories()
    dc.pp.filter_by_expr(adata_sub, group="cluster", min_count=10, min_total_count=15)

    dds = DeseqDataSet(
        adata=adata_sub,
        design="~ cluster",
        refit_cooks=True
    )

    dds.deseq2()
    results[celltype] = {}

    for g1, g2 in feasible_contrasts:
        contrast_name = f"{g1}_vs_{g2}".replace(" ", "")
        print(f"  Running contrast: {g1} vs {g2}")

        stat = DeseqStats(
            dds,
            contrast=["cluster", g1, g2]
        )
        stat.summary()
        results[celltype][contrast_name] = stat.results_df

    for name, df in results[celltype].items():
        df.to_csv(os.path.join(outdir, f"{celltype}_{name}.csv"))

In [ ]:
pdata.obs.groupby(['auto_cell_type_V9', 'cluster']).size().unstack(fill_value=0)


In [ ]:
gene_sets = gp.read_gmt("/Users/boscens/Desktop/reznik/bodycomp_main/data/reference/h.all.v2025.1.Hs.symbols.gmt")

In [ ]:
def run_gsea_for_all(results, gene_sets):

    all_results = []

    for ct, ct_res in results.items():
        print(f"Running GSEA for: {ct}")

        for comp, df in ct_res.items():

            valid = df[df["padj"].notna()]
            ranks = valid["stat"].dropna().sort_values(ascending=False)
            print(ranks.head())

            pre = gp.prerank(
                rnk=ranks,
                gene_sets=gene_sets,
                permutation_num=1000,
                seed=42,
                outdir=None,
                no_plot=True
            )

            res = pre.res2d.copy()

            res["cell_type"] = ct
            res["comparison"] = comp
            res["pathway"] = res.index

            all_results.append(res.reset_index(drop=True))

    final_df = pd.concat(all_results, ignore_index=True)

    return final_df

In [ ]:
gsea_results = run_gsea_for_all(results, gene_sets)
gsea_results.to_csv(os.path.expanduser("~/Desktop/reznik/bodycomp_main/revision/results/polar_scrna/gsea_all_results_pairwise_0907.csv"), index=False)